<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_5_Specialization/Track_4_Computer_Vision/object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection with YOLO and SSD
## Learning Objectives
- Understand object detection vs. classification
- Work with bounding boxes and IoU metrics
- Use YOLOv8 for real-time object detection


In [ ]:
# Detection concepts
detection_concepts = {
    'Classification': 'What is in the image?',
    'Localization': 'Where is the object? (single object)',
    'Detection': 'What and where? (multiple objects)',
    'Segmentation': 'Pixel-level mask for each object',
    'Bounding Box': '[x_min, y_min, x_max, y_max] coordinates',
    'IoU': 'Intersection over Union - measures box overlap',
    'NMS': 'Non-Maximum Suppression - removes duplicate detections',
    'mAP': 'mean Average Precision - standard detection metric'
}
for k, v in detection_concepts.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Implementing IoU from scratch
import numpy as np

def compute_iou(box1, box2):
    """Compute IoU between two bounding boxes [x1, y1, x2, y2]."""
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - intersection
    return intersection / union

pred = [50, 50, 200, 200]
gt   = [60, 60, 210, 210]
print(f'Predicted box: {pred}')
print(f'Ground truth:  {gt}')
print(f'IoU: {compute_iou(pred, gt):.4f}')

In [ ]:
# YOLOv8 usage (pip install ultralytics)
yolo_code = '''
from ultralytics import YOLO
import cv2

# Load pretrained model
model = YOLO('yolov8n.pt')  # nano model (fastest)

# Inference on image
results = model('bus.jpg')

# Process results
for result in results:
    boxes = result.boxes  # bounding boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        cls = box.cls[0].item()
        label = model.names[int(cls)]
        print(f"{label}: conf={conf:.2f}, box=[{x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f}]")

# Save annotated image
results[0].save(filename='result.jpg')
'''
print('YOLOv8 inference code:')
print(yolo_code)

In [ ]:
# Non-Maximum Suppression from scratch
def nms(boxes, scores, iou_threshold=0.5):
    """Simple NMS implementation."""
    indices = np.argsort(scores)[::-1]
    keep = []
    while len(indices) > 0:
        best = indices[0]
        keep.append(best)
        rest = indices[1:]
        ious = np.array([compute_iou(boxes[best], boxes[i]) for i in rest])
        indices = rest[ious < iou_threshold]
    return keep

# Example
boxes_ex = [[10,10,50,50],[12,12,52,52],[100,100,150,150],[102,102,152,152]]
scores_ex = [0.9, 0.75, 0.85, 0.6]
kept = nms(boxes_ex, scores_ex)
print(f'After NMS, keeping {len(kept)} of {len(boxes_ex)} boxes: indices {kept}')

## Exercises
1. Run YOLOv8 on a local image and visualize detections.
2. Fine-tune YOLOv8 on a custom dataset with 100+ images.
3. Implement NMS and benchmark it against `torchvision.ops.nms`.


## Summary
- Object detection combines classification and localization.
- IoU measures bounding box overlap; NMS removes duplicates.
- YOLO achieves real-time detection with a single network pass.
